In [ ]:
# FireWatch AI: Fire and Smoke Inference

In this notebook, I will use my already-trained YOLO26n fire/smoke model to detect fire and smoke in images and prerecorded videos.

I will not train the model again. My goal is to load the existing best checkpoint, run inference, inspect the detections, and save annotated outputs for later integration into the FireWatch AI application.

I will keep the PPE module on hold for future development and focus on the fire/smoke pipeline first.


In [11]:
# I am connecting Google Drive and checking the trained model.
# This step only verifies the saved checkpoint. It does not train anything.

from pathlib import Path
from google.colab import drive

# Mount Drive so I can access the FireWatch AI files.
drive.mount("/content/drive", force_remount=False)

# Define the project and existing fire/smoke model paths.
PROJECT_ROOT = Path("/content/drive/MyDrive/FireWatch-AI")

BEST_CHECKPOINT = (
    PROJECT_ROOT
    / "runs/fire_smoke/yolo26n_fire_smoke_v1/weights/best.pt"
)

print("Project folder exists:", PROJECT_ROOT.is_dir())
print("Trained checkpoint exists:", BEST_CHECKPOINT.is_file())

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError("FireWatch AI project folder was not found.")

if not BEST_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"Trained fire/smoke checkpoint was not found: {BEST_CHECKPOINT}"
    )

print("\nCheckpoint verification: PASS")
print("Training started: NO")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder exists: True
Trained checkpoint exists: True

Checkpoint verification: PASS
Training started: NO


In [12]:
from pathlib import Path
from google.colab import drive

# Connect Drive and locate my trained fire/smoke model.
drive.mount("/content/drive", force_remount=False)

PROJECT = Path("/content/drive/MyDrive/FireWatch-AI")
MODEL_PATH = PROJECT / "runs/fire_smoke/yolo26n_fire_smoke_v1/weights/best.pt"

print("Project exists:", PROJECT.is_dir())
print("Model exists:", MODEL_PATH.is_file())

assert PROJECT.is_dir(), "FireWatch AI project folder not found."
assert MODEL_PATH.is_file(), "Trained model checkpoint not found."

# Save future inference outputs separately.
OUTPUT_DIR = PROJECT / "outputs/fire_smoke_inference"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output folder:", OUTPUT_DIR)
print("Checkpoint verification: PASS")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project exists: True
Model exists: True
Output folder: /content/drive/MyDrive/FireWatch-AI/outputs/fire_smoke_inference
Checkpoint verification: PASS


In [4]:
%pip install -q ultralytics

In [5]:
from ultralytics import YOLO

# Load the trained checkpoint saved in Google Drive.
model = YOLO(str(MODEL_PATH))

print("Model loaded successfully.")
print("Classes:", model.names)
print("Inference output folder:", OUTPUT_DIR)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Model loaded successfully.
Classes: {0: 'fire', 1: 'smoke'}
Inference output folder: /content/drive/MyDrive/FireWatch-AI/outputs/fire_smoke_inference


In [6]:
from pathlib import Path

# Find one image from the prepared fire/smoke test dataset.
DATASET_DIR = PROJECT / "datasets/candidates/fire_smoke_v2_bbox_candidate"

image_extensions = {".jpg", ".jpeg", ".png", ".webp"}
image_paths = [
    p for p in (DATASET_DIR / "test/images").iterdir()
    if p.suffix.lower() in image_extensions
]

assert image_paths, "No test images found in the expected folder."

sample_image = image_paths[0]

# Run inference and save the annotated image.
results = model.predict(
    source=str(sample_image),
    conf=0.25,
    save=True,
    project=str(OUTPUT_DIR),
    name="sample_image",
    exist_ok=True,
    verbose=False
)

print("Input image:", sample_image.name)
print("Annotated output folder:", OUTPUT_DIR / "sample_image")

for result in results:
    for box in result.boxes:
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])
        print(f"{model.names[class_id]}: {confidence:.3f}")

Results saved to /content/drive/MyDrive/FireWatch-AI/outputs/fire_smoke_inference/sample_image
Input image: 908_jpg.rf.95f56ccbb827bc41cf502e0785fb0423.jpg
Annotated output folder: /content/drive/MyDrive/FireWatch-AI/outputs/fire_smoke_inference/sample_image
fire: 0.614


In [9]:
from pathlib import Path
import yaml

# Locate and read the prepared fire/smoke dataset configuration.
DATASET_DIR = PROJECT / "datasets/candidates/fire_smoke_v2_bbox_candidate"
DATA_YAML = DATASET_DIR / "data.yaml"

assert DATA_YAML.is_file(), f"Dataset YAML not found: {DATA_YAML}"

with open(DATA_YAML, "r") as file:
    config = yaml.safe_load(file)

print("Dataset YAML:", DATA_YAML)
print("Classes:", config.get("names"))

# Check the configured train, validation and test folders for videos.
dataset_root = Path(config.get("path", DATASET_DIR))
if not dataset_root.is_absolute():
    dataset_root = DATASET_DIR / dataset_root

video_extensions = {".mp4", ".avi", ".mov", ".mkv"}

for split in ("train", "val", "test"):
    split_value = config.get(split)

    if not split_value:
        print(f"\n{split}: no path defined")
        continue

    split_path = Path(split_value)
    if not split_path.is_absolute():
        split_path = dataset_root / split_path

    # Image datasets commonly store files inside an images subfolder.
    search_path = split_path
    if split_path.name != "images" and (split_path / "images").is_dir():
        search_path = split_path / "images"

    videos = [
        p for p in search_path.rglob("*")
        if p.is_file() and p.suffix.lower() in video_extensions
    ] if search_path.is_dir() else []

    print(f"\n{split}: {search_path}")
    print("Folder exists:", search_path.is_dir())
    print("Videos found:", len(videos))
    for video in videos[:5]:
        print(" -", video.name)

Dataset YAML: /content/drive/MyDrive/FireWatch-AI/datasets/candidates/fire_smoke_v2_bbox_candidate/data.yaml
Classes: ['fire', 'smoke']

train: /content/drive/MyDrive/FireWatch-AI/datasets/candidates/fire_smoke_v2_bbox_candidate/train/images
Folder exists: True
Videos found: 0

val: /content/drive/MyDrive/FireWatch-AI/datasets/candidates/fire_smoke_v2_bbox_candidate/valid/images
Folder exists: True
Videos found: 0

test: /content/drive/MyDrive/FireWatch-AI/datasets/candidates/fire_smoke_v2_bbox_candidate/test/images
Folder exists: True
Videos found: 0


In [ ]:
## Summary

- Loaded the trained YOLO26n fire/smoke model.
- Successfully ran inference on one test image.
- Detected fire with 61.4% confidence.
- Saved the annotated image in `outputs/fire_smoke_inference/sample_image/`.
- Confirmed that the configured dataset splits contain images, not videos.

### Pending
- Test on additional images.
- Video inference when sample footage is available.
- Temporal analysis and fire risk scoring.
- Streamlit dashboard integration.

### Next step
Continue image inference testing using the existing trained model.

In [10]:
# I will test the trained model on five test images.
image_paths = sorted((DATASET_DIR / "test/images").glob("*"))
image_paths = [p for p in image_paths if p.suffix.lower() in image_extensions][:5]

results = model.predict(
    source=[str(p) for p in image_paths],
    conf=0.25,
    save=True,
    project=str(OUTPUT_DIR),
    name="five_image_test",
    exist_ok=True,
    verbose=False
)

for path, result in zip(image_paths, results):
    detections = [
        f"{model.names[int(box.cls[0])]} ({float(box.conf[0]):.2f})"
        for box in result.boxes
    ]
    print(f"{path.name}: {', '.join(detections) if detections else 'No detection'}")

print("\nSaved outputs:", OUTPUT_DIR / "five_image_test")

Results saved to /content/drive/MyDrive/FireWatch-AI/outputs/fire_smoke_inference/five_image_test
000026_jpg.rf.4e32ef25e025caef2f6852167f7dbd1a.jpg: smoke (0.90)
000040_jpg.rf.97067f36eea66e3582fb3ac455095e79.jpg: smoke (0.90)
000070_jpg.rf.e09774c4cab08f84861a0cffe742e1a1.jpg: smoke (0.87)
000093_png_jpg.rf.96fee0cfc7c9ba34be520d35efc9ad00.jpg: smoke (0.84)
000117_jpg.rf.c920477f7cd04ff1b142a5485df54019.jpg: smoke (0.91), smoke (0.90), smoke (0.85), smoke (0.83), smoke (0.68), smoke (0.31)

Saved outputs: /content/drive/MyDrive/FireWatch-AI/outputs/fire_smoke_inference/five_image_test


In [14]:
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive")

matches = list(DRIVE.rglob("smoke_videos.1407.zip"))

for path in matches:
    print("ZIP found:", path)

if not matches:
    print("ZIP not found. Check that Google Drive is mounted and upload is complete.")


ZIP found: /content/drive/MyDrive/FireWatch-AI/datasets/videos/firesense/smoke_videos.1407.zip


In [15]:
from pathlib import Path
import zipfile

ZIP_PATH = Path("/content/drive/MyDrive/FireWatch-AI/datasets/videos/firesense/smoke_videos.1407.zip")
EXTRACT_DIR = Path("/content/drive/MyDrive/FireWatch-AI/datasets/videos/firesense/smoke")

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(EXTRACT_DIR)

videos = list(EXTRACT_DIR.rglob("*.avi"))
print("Videos extracted:", len(videos))
print("Sample videos:", [p.name for p in videos[:5]])

Videos extracted: 22
Sample videos: ['testneg01.807.avi', 'testneg02.808.avi', 'testneg03.809.avi', 'testneg04.810.avi', 'testneg05.811.avi']


In [16]:
import requests
from pathlib import Path

url = "https://zenodo.org/records/836749/files/fire_videos.1406.zip"
dest = Path("/content/drive/MyDrive/FireWatch-AI/datasets/videos/firesense/fire_videos.1406.zip")
dest.parent.mkdir(parents=True, exist_ok=True)

with requests.get(url, stream=True, timeout=(20, 120)) as r:
    r.raise_for_status()
    with open(dest, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

print("Download complete:", dest.exists())
print("File size (MB):", round(dest.stat().st_size / (1024 * 1024), 1))

Download complete: True
File size (MB): 593.8


In [17]:
from pathlib import Path
import zipfile

ZIP_PATH = Path("/content/drive/MyDrive/FireWatch-AI/datasets/videos/firesense/fire_videos.1406.zip")
EXTRACT_DIR = ZIP_PATH.parent / "fire"

with zipfile.ZipFile(ZIP_PATH) as z:
    bad_file = z.testzip()
    assert bad_file is None, f"Corrupt file: {bad_file}"
    z.extractall(EXTRACT_DIR)

videos = [p for p in EXTRACT_DIR.rglob("*")
          if p.is_file() and p.suffix.lower() in {".avi", ".mp4", ".mov", ".mkv"}]

print("ZIP integrity: PASS")
print("Fire videos extracted:", len(videos))
print("Sample videos:", [p.name for p in videos[:5]])

ZIP integrity: PASS
Fire videos extracted: 27
Sample videos: ['negsVideo1.858.avi', 'negsVideo2.859.avi', 'negsVideo3.860.avi', 'negsVideo4.861.avi', 'negsVideo5.862.avi']


In [18]:
from pathlib import Path

VIDEO_ROOT = PROJECT / "datasets/videos/firesense"
VIDEO_EXTS = {".avi", ".mp4", ".mov", ".mkv"}

for category in ["fire", "smoke"]:
    root = VIDEO_ROOT / category
    videos = [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTS]
    print(f"\n{category.upper()}: {len(videos)} videos")

    for label in ["pos", "neg"]:
        matches = [p for p in videos if label in [part.lower() for part in p.parts]]
        print(f"{label}: {len(matches)}")
        if matches:
            print("Sample:", matches[0].name)


FIRE: 27 videos
pos: 11
Sample: posVideo1.868.avi
neg: 16
Sample: negsVideo1.858.avi

SMOKE: 22 videos
pos: 13
Sample: testpos01.817.avi
neg: 9
Sample: testneg01.807.avi


In [19]:
from pathlib import Path
from collections import Counter

VIDEO_ROOT = PROJECT / "datasets/videos/firesense"
VIDEO_EXTS = {".avi", ".mp4", ".mov", ".mkv"}

fire_video = sorted((VIDEO_ROOT / "fire" / "pos").glob("*.avi"))[0]
smoke_video = sorted((VIDEO_ROOT / "smoke" / "pos").glob("*.avi"))[0]

def test_video(path, name):
    counts, confidence, frames = Counter(), Counter(), 0

    results = model.predict(
        source=str(path),
        conf=0.25,
        vid_stride=10,
        stream=True,
        save=True,
        project=str(OUTPUT_DIR),
        name=name,
        exist_ok=True,
        verbose=False
    )

    for result in results:
        frames += 1
        for box in result.boxes:
            cls = model.names[int(box.cls[0])]
            counts[cls] += 1
            confidence[cls] += float(box.conf[0])

    print(f"\n{name}: {path.name}")
    print("Frames processed:", frames)
    print("Detections:", dict(counts))
    print("Average confidence:", {
        cls: round(confidence[cls] / counts[cls], 3)
        for cls in counts
    })
    print("Saved to:", OUTPUT_DIR / name)

test_video(fire_video, "fire_video_test")
test_video(smoke_video, "smoke_video_test")

Results saved to /content/drive/MyDrive/FireWatch-AI/outputs/fire_smoke_inference/fire_video_test

fire_video_test: posVideo1.868.avi
Frames processed: 29
Detections: {'fire': 114}
Average confidence: {'fire': 0.491}
Saved to: /content/drive/MyDrive/FireWatch-AI/outputs/fire_smoke_inference/fire_video_test
Results saved to /content/drive/MyDrive/FireWatch-AI/outputs/fire_smoke_inference/smoke_video_test

smoke_video_test: testpos01.817.avi
Frames processed: 40
Detections: {'smoke': 26, 'fire': 1}
Average confidence: {'smoke': 0.737, 'fire': 0.343}
Saved to: /content/drive/MyDrive/FireWatch-AI/outputs/fire_smoke_inference/smoke_video_test


In [ ]:
## Summary

### Completed
- Loaded the existing trained YOLO26n fire/smoke model.
- Tested inference on one image and five additional test images.
- Saved annotated image outputs.
- Extracted FIRESENSE fire and smoke video archives.
- Successfully tested one positive fire video and one positive smoke video.

### Verified Results
- Fire video: 29 sampled frames, 114 fire detections.
- Smoke video: 40 sampled frames, 26 smoke detections and 1 fire detection.
- Existing model checkpoint was reused. No retraining was performed.